# SalesPrep — exploration

Parcours complet des étapes 1.a à 7 : chargement ventes, agrégations, imputations, pivots et jointure.

Remplit `../Output/` avec chaque dataframe intermédiaire.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "SalesPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Entrées — ventes brutes et lookup hôtel

In [2]:
from rod_ia.config.settings import get_settings
from prepare._shared.sales_base import load_sales_frame

settings = get_settings(PROJECT)
sales_path = settings.sales_csv_path
holdout_year = 2026

sales_raw_head = pd.read_csv(sales_path, nrows=8)
print("Aperçu fichier ventes (8 lignes) :")
sales_raw_head

Aperçu fichier ventes (8 lignes) :


,NOM BOUTIQUE,OPERATEUR,MACHINE,DATE,HEURE,STATUT,CODE EAN,NOM DU PRODUIT,QUANTITE,PRIX HT,VAT,PRIX TTC,TYPE,GAMME,MARQUE,FOURNISSEUR,ORDER ID (TICKET DE CAISSE),TEMPERATURE,METEO DU JOUR (MOYENNE),METEO DU MOIS (MOYENNE)
0,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,12:04:22.555,DONE,3583787508251,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9281,23.1,NaN,NaN
1,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-10,18:32:38.901,DONE,3608439285448,CASQUETTE ENFANT -MH100,1,NaN,20.0,12.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9282,23.1,NaN,NaN
2,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-11,12:11:15.59,DONE,3583788165002,MASQUE EASYBREATH DE SURFACE ADULTE - 500 BLEU,1,NaN,20.0,29.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9283,24.6,NaN,NaN
3,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-12,12:50:46.708,DONE,3583787810484,LUNETTES DE NATATION VERRES CLAIRS XBASE TAILL...,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9284,26.4,NaN,NaN
4,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-12,17:00:42.96,DONE,3583787229026,BOARDSHORT HENDAIA ECO NT BLEU,1,NaN,20.0,9.0,NON-F&B,PAP,DECATHLON,DECATHLON,9285,26.4,NaN,NaN
5,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-13,11:40:04.761,DONE,3583787508251,TONGS FEMME 100 NOIR,1,NaN,20.0,6.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9286,25.4,NaN,NaN
6,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-13,13:43:48.323,DONE,3583787791523,MF COMPACT XL TOWEL BLUE PETROL,1,NaN,20.0,15.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9287,25.4,NaN,NaN
7,Ibis budget Nice,ADIPOS,SCANNER NICE,2023-08-13,16:15:52.677,DONE,3583787791523,MF COMPACT XL TOWEL BLUE PETROL,1,NaN,20.0,15.0,NON-F&B,ACCESSOIRES,DECATHLON,DECATHLON,9288,25.4,NaN,NaN


In [3]:
lookup_path = ROD_OUTPUT / "hotel_lookup.parquet"
rod_lookup = pd.read_parquet(lookup_path) if lookup_path.exists() else None
if rod_lookup is not None:
    rod_lookup = rod_lookup[["nom_hotel", "hotel_code"]].drop_duplicates()
    print(f"Lookup RodPrep : {len(rod_lookup)} hôtels")
    rod_lookup.head()
else:
    print("Exécuter d'abord RodPrep/Explore/explore.ipynb")

Lookup RodPrep : 8 hôtels


## 2. Phase 0 — normalisation ligne à ligne

In [4]:
raw = load_sales_frame(sales_path, exclude_year=holdout_year)
if rod_lookup is not None:
    raw = raw.merge(rod_lookup, on="nom_hotel", how="left")
    raw["hotel_code"] = raw["hotel_code"].fillna(raw["nom_hotel"])
else:
    raw["hotel_code"] = raw["nom_hotel"]

print(f"Lignes retenues (annee < {holdout_year}) : {len(raw):,}")
raw[["nom_hotel", "hotel_code", "annee", "mois", "categorie", "sous_categorie",
     "nombre_ventes", "montant_ventes", "heure_vente", "is_weekend", "is_holiday"]].head(8)

Lignes retenues (annee < 2026) : 73,296


,nom_hotel,hotel_code,annee,mois,categorie,sous_categorie,nombre_ventes,montant_ventes,heure_vente,is_weekend,is_holiday
0,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,6.0,12,0,0
1,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,12.0,18,0,0
2,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,29.0,12,0,0
3,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,6.0,12,1,0
4,Ibis budget Nice,H2075,2023,8,N_F_B,PAP,1,9.0,17,1,0
5,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,6.0,11,1,0
6,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,15.0,13,1,0
7,Ibis budget Nice,H2075,2023,8,N_F_B,ACCESSOIRES,1,15.0,16,1,0


## 3. Étape 1 — agrégation annuelle

In [5]:
from sales_prep import aggregations as agg

step_1a = agg.step_1a_annual_raw(raw)
step_1b = agg.step_1b_annual_normalized(step_1a)
step_1c = agg.step_1c_annual_divided_by_12(step_1a)

print("1.a — brut annuel")
step_1a.head()

1.a — brut annuel


,nom_hotel,annee,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits,nombre_categories_annee_f_b,nombre_categories_annee_n_f_b,pct_categories_annee_f_b,pct_categories_annee_n_f_b,nombre_mois,premier_mois,dernier_mois,mois_actifs,mois_manquants
0,Ibis budget Nice,2023,94.0,1012.0,80.0,26.0,0.0,4.0,0.000000,1.000000,4,8,11,4,8
1,Ibis budget Nice,2024,1690.0,5356.1,1097.0,88.0,3.0,5.0,0.375000,0.625000,10,3,12,10,2
2,Ibis budget Nice,2025,3678.0,12048.7,2441.0,105.0,3.0,6.0,0.333333,0.666667,12,1,12,12,0
3,Ibis budget Strasbourg Centre République,2024,786.0,2897.5,527.0,34.0,3.0,0.0,1.000000,0.000000,3,10,12,3,9
4,Ibis budget Strasbourg Centre République,2025,4122.0,13991.5,2646.0,32.0,3.0,0.0,1.000000,0.000000,12,1,12,12,0


In [6]:
print("1.b — annualisé (÷ mois_actifs × 12)")
step_1b.head()

1.b — annualisé (÷ mois_actifs × 12)


,nom_hotel,annee,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits,nombre_categories_annee_f_b,nombre_categories_annee_n_f_b,pct_categories_annee_f_b,pct_categories_annee_n_f_b,nombre_mois,premier_mois,dernier_mois,mois_actifs,mois_manquants
0,Ibis budget Nice,2023,282.0,3036.00,240.0,78.0,0.0,4.0,0.000000,1.000000,4,8,11,4,8
1,Ibis budget Nice,2024,2028.0,6427.32,1316.4,105.6,3.0,5.0,0.375000,0.625000,10,3,12,10,2
2,Ibis budget Nice,2025,3678.0,12048.70,2441.0,105.0,3.0,6.0,0.333333,0.666667,12,1,12,12,0
3,Ibis budget Strasbourg Centre République,2024,3144.0,11590.00,2108.0,136.0,3.0,0.0,1.000000,0.000000,3,10,12,3,9
4,Ibis budget Strasbourg Centre République,2025,4122.0,13991.50,2646.0,32.0,3.0,0.0,1.000000,0.000000,12,1,12,12,0


In [7]:
print("1.c — divisé par 12")
step_1c.head()

1.c — divisé par 12


,nom_hotel,annee,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits
0,Ibis budget Nice,2023,7.833333,84.333333,6.666667,2.166667
1,Ibis budget Nice,2024,140.833333,446.341667,91.416667,7.333333
2,Ibis budget Nice,2025,306.500000,1004.058333,203.416667,8.750000
3,Ibis budget Strasbourg Centre République,2024,65.500000,241.458333,43.916667,2.833333
4,Ibis budget Strasbourg Centre République,2025,343.500000,1165.958333,220.500000,2.666667


## 4. Étape 2 — agrégation mensuelle + imputation

In [8]:
step_2a = agg.step_2a_monthly_raw(raw)
step_2b = agg.step_2b_monthly_imputed(step_2a)

print(f"2.a : {len(step_2a)} lignes | 2.b : {len(step_2b)} lignes (après imputation)")
step_2a.head(8)

2.a : 122 lignes | 2.b : 155 lignes (après imputation)


,nom_hotel,annee,mois,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits,nombre_categories_mois_f_b,nombre_categories_mois_n_f_b,pct_categories_mois_f_b,pct_categories_mois_n_f_b,nombre_mois,premier_mois,dernier_mois,mois_actifs,mois_manquants
0,Ibis budget Nice,2023,8,41.0,475.0,33.0,18.0,0.0,4.0,0.000000,1.000000,4,8,11,4,8
1,Ibis budget Nice,2023,9,30.0,322.0,27.0,16.0,0.0,4.0,0.000000,1.000000,4,8,11,4,8
2,Ibis budget Nice,2023,10,22.0,206.0,19.0,11.0,0.0,2.0,0.000000,1.000000,4,8,11,4,8
3,Ibis budget Nice,2023,11,1.0,9.0,1.0,1.0,0.0,1.0,0.000000,1.000000,4,8,11,4,8
4,Ibis budget Nice,2024,3,2.0,16.0,2.0,2.0,0.0,1.0,0.000000,1.000000,10,3,12,10,2
5,Ibis budget Nice,2024,4,12.0,145.0,12.0,7.0,0.0,3.0,0.000000,1.000000,10,3,12,10,2
6,Ibis budget Nice,2024,5,53.0,270.8,44.0,24.0,1.0,2.0,0.333333,0.666667,10,3,12,10,2
7,Ibis budget Nice,2024,6,247.0,841.5,177.0,40.0,2.0,5.0,0.285714,0.714286,10,3,12,10,2


In [9]:
step_2b.sort_values(["nom_hotel", "annee", "mois"]).head(12)

,nom_hotel,annee,mois,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits,nombre_categories_mois_f_b,nombre_categories_mois_n_f_b,pct_categories_mois_f_b,pct_categories_mois_n_f_b,nombre_mois,premier_mois,dernier_mois,mois_actifs,mois_manquants
122,Ibis budget Nice,2023,1,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
123,Ibis budget Nice,2023,2,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
124,Ibis budget Nice,2023,3,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
125,Ibis budget Nice,2023,4,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
126,Ibis budget Nice,2023,5,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
127,Ibis budget Nice,2023,6,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
128,Ibis budget Nice,2023,7,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,8
0,Ibis budget Nice,2023,8,41.0,475.0,33.0,18.0,0.0,4.00,0.0,1.0,4,8,11,4,8
1,Ibis budget Nice,2023,9,30.0,322.0,27.0,16.0,0.0,4.00,0.0,1.0,4,8,11,4,8
2,Ibis budget Nice,2023,10,22.0,206.0,19.0,11.0,0.0,2.00,0.0,1.0,4,8,11,4,8


## 5. Étape 3 — catégorie / sous-catégorie

In [10]:
step_3a = agg.step_3a_category_monthly(raw)
step_3b = agg.step_3b_category_imputed(step_3a, step_2a)
step_3c = agg.step_3c_category_wide(step_3b)

print(f"3.a: {step_3a.shape} | 3.b: {step_3b.shape} | 3.c: {step_3c.shape}")
step_3a.head(8)

3.a: (699, 9) | 3.b: (862, 9) | 3.c: (155, 63)


,nom_hotel,annee,mois,categorie,sous_categorie,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits
0,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,32,383.0,25,11
1,Ibis budget Nice,2023,8,N_F_B,COSMETIQUE,1,10.0,1,1
2,Ibis budget Nice,2023,8,N_F_B,PAP,7,74.0,7,5
3,Ibis budget Nice,2023,8,N_F_B,SOS,1,8.0,1,1
4,Ibis budget Nice,2023,9,N_F_B,ACCESSOIRES,16,186.0,13,9
5,Ibis budget Nice,2023,9,N_F_B,COSMETIQUE,3,30.0,3,1
6,Ibis budget Nice,2023,9,N_F_B,PAP,10,98.0,10,5
7,Ibis budget Nice,2023,9,N_F_B,SOS,1,8.0,1,1


In [11]:
step_3c.head()

,nom_hotel,annee,mois,cat_f_b_nombre_ventes,cat_nan_nombre_ventes,cat_n_f_b_nombre_ventes,cat_f_b_montant_ventes,cat_nan_montant_ventes,cat_n_f_b_montant_ventes,cat_f_b_nombre_paniers,cat_nan_nombre_paniers,cat_n_f_b_nombre_paniers,cat_f_b_nombre_produits,cat_nan_nombre_produits,cat_n_f_b_nombre_produits,...,sous_cat_sos_nombre_paniers,sous_cat_souvenirs_nombre_paniers,sous_cat_nan_nombre_paniers,sous_cat_ref_nombre_produits,sous_cat_accessoires_nombre_produits,sous_cat_alcool_nombre_produits,sous_cat_cosmetique_nombre_produits,sous_cat_food_salee_nombre_produits,sous_cat_food_sucree_nombre_produits,sous_cat_jeux_enfants_nombre_produits,sous_cat_pap_nombre_produits,sous_cat_sans_alcool_nombre_produits,sous_cat_sos_nombre_produits,sous_cat_souvenirs_nombre_produits,sous_cat_nan_nombre_produits
0,Ibis budget Nice,2023,1,0.0,0.0,23.5,0.0,0.0,253.0,0.0,0.0,20.25,0.0,0.0,11.5,...,0.5,0.0,0.0,0.0,7.0,0.0,0.5,0.0,0.0,0.0,3.5,0.0,0.5,0.0,0.0
1,Ibis budget Nice,2023,2,0.0,0.0,23.5,0.0,0.0,253.0,0.0,0.0,20.25,0.0,0.0,11.5,...,0.5,0.0,0.0,0.0,7.0,0.0,0.5,0.0,0.0,0.0,3.5,0.0,0.5,0.0,0.0
2,Ibis budget Nice,2023,3,0.0,0.0,23.5,0.0,0.0,253.0,0.0,0.0,20.25,0.0,0.0,11.5,...,0.5,0.0,0.0,0.0,7.0,0.0,0.5,0.0,0.0,0.0,3.5,0.0,0.5,0.0,0.0
3,Ibis budget Nice,2023,4,0.0,0.0,23.5,0.0,0.0,253.0,0.0,0.0,20.25,0.0,0.0,11.5,...,0.5,0.0,0.0,0.0,7.0,0.0,0.5,0.0,0.0,0.0,3.5,0.0,0.5,0.0,0.0
4,Ibis budget Nice,2023,5,0.0,0.0,23.5,0.0,0.0,253.0,0.0,0.0,20.25,0.0,0.0,11.5,...,0.5,0.0,0.0,0.0,7.0,0.0,0.5,0.0,0.0,0.0,3.5,0.0,0.5,0.0,0.0


## 6. Étape 4 — heure de vente

In [12]:
step_4a = agg.step_4a_hourly(step_3b, raw)
step_4b = agg.step_4b_hourly_imputed(step_4a, step_2a)
step_4c = agg.step_4c_hourly_wide(step_4b)
print(f"4.a: {step_4a.shape} | 4.b: {step_4b.shape} | 4.c: {step_4c.shape}")
step_4a.head(8)

4.a: (7332, 10) | 4.b: (8036, 10) | 4.c: (155, 99)


,nom_hotel,annee,mois,categorie,sous_categorie,heure_vente,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits
0,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,10,3,59.0,2,3
1,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,11,3,25.0,3,3
2,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,12,5,60.0,5,4
3,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,13,1,15.0,1,1
4,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,14,1,12.0,1,1
5,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,15,1,15.0,1,1
6,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,16,4,54.0,2,2
7,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,17,3,41.0,2,2


## 7. Étape 5 — week-end

In [13]:
step_5a = agg.step_5a_weekend(raw)
step_5b = agg.step_5b_weekend_imputed(step_5a, step_2a)
step_5c = agg.step_5c_weekend_wide(step_5b)
print(f"5.a: {step_5a.shape} | 5.c: {step_5c.shape}")
step_5a.head(8)

5.a: (1253, 10) | 5.c: (155, 11)


,nom_hotel,annee,mois,categorie,sous_categorie,is_weekend,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits
0,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,0,22,265.0,16,11
1,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,1,10,118.0,9,5
2,Ibis budget Nice,2023,8,N_F_B,COSMETIQUE,0,1,10.0,1,1
3,Ibis budget Nice,2023,8,N_F_B,PAP,0,5,52.0,5,5
4,Ibis budget Nice,2023,8,N_F_B,PAP,1,2,22.0,2,2
5,Ibis budget Nice,2023,8,N_F_B,SOS,0,1,8.0,1,1
6,Ibis budget Nice,2023,9,N_F_B,ACCESSOIRES,0,8,87.0,6,5
7,Ibis budget Nice,2023,9,N_F_B,ACCESSOIRES,1,8,99.0,7,6


## 8. Étape 6 — jour férié

In [14]:
step_6a = agg.step_6a_holiday(raw)
step_6b = agg.step_6b_holiday_imputed(step_6a, step_2a)
step_6c = agg.step_6c_holiday_wide(step_6b)
print(f"6.a: {step_6a.shape} | 6.c: {step_6c.shape}")
step_6a.head(8)

6.a: (859, 10) | 6.c: (155, 11)


,nom_hotel,annee,mois,categorie,sous_categorie,is_holiday,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits
0,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,0,31,370.0,24,11
1,Ibis budget Nice,2023,8,N_F_B,ACCESSOIRES,1,1,13.0,1,1
2,Ibis budget Nice,2023,8,N_F_B,COSMETIQUE,0,1,10.0,1,1
3,Ibis budget Nice,2023,8,N_F_B,PAP,0,7,74.0,7,5
4,Ibis budget Nice,2023,8,N_F_B,SOS,0,1,8.0,1,1
5,Ibis budget Nice,2023,9,N_F_B,ACCESSOIRES,0,16,186.0,13,9
6,Ibis budget Nice,2023,9,N_F_B,COSMETIQUE,0,3,30.0,3,1
7,Ibis budget Nice,2023,9,N_F_B,PAP,0,10,98.0,10,5


## 9. Étape 7 — jointure finale

In [15]:
from sales_prep.pipeline import SalesPrep

joined = SalesPrep._join_all(
    [step_2b, step_3c, step_4c, step_5c, step_6c],
    keys=["nom_hotel", "hotel_code", "annee", "mois"],
)
print(f"Jointure : {joined.shape}")
joined.head()

Jointure : (155, 188)


,nom_hotel,annee,mois,nombre_ventes,montant_ventes,nombre_paniers,nombre_produits,nombre_categories_mois_f_b,nombre_categories_mois_n_f_b,pct_categories_mois_f_b,pct_categories_mois_n_f_b,nombre_mois,premier_mois,dernier_mois,mois_actifs,...,weekend_1_nombre_ventes,weekend_0_montant_ventes,weekend_1_montant_ventes,weekend_0_nombre_paniers,weekend_1_nombre_paniers,weekend_0_nombre_produits,weekend_1_nombre_produits,holiday_0_nombre_ventes,holiday_1_nombre_ventes,holiday_0_montant_ventes,holiday_1_montant_ventes,holiday_0_nombre_paniers,holiday_1_nombre_paniers,holiday_0_nombre_produits,holiday_1_nombre_produits
0,Ibis budget Nice,2023,1,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,...,8.5,160.75,92.25,12.25,8.0,9.25,6.25,23.25,0.25,249.75,3.25,20.0,0.25,11.5,0.25
1,Ibis budget Nice,2023,2,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,...,8.5,160.75,92.25,12.25,8.0,9.25,6.25,23.25,0.25,249.75,3.25,20.0,0.25,11.5,0.25
2,Ibis budget Nice,2023,3,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,...,8.5,160.75,92.25,12.25,8.0,9.25,6.25,23.25,0.25,249.75,3.25,20.0,0.25,11.5,0.25
3,Ibis budget Nice,2023,4,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,...,8.5,160.75,92.25,12.25,8.0,9.25,6.25,23.25,0.25,249.75,3.25,20.0,0.25,11.5,0.25
4,Ibis budget Nice,2023,5,23.5,253.0,20.0,11.5,0.0,2.75,0.0,1.0,4,8,11,4,...,8.5,160.75,92.25,12.25,8.0,9.25,6.25,23.25,0.25,249.75,3.25,20.0,0.25,11.5,0.25


## 10. Persistance Output/

In [16]:
artifacts = {
    "step_1a": step_1a, "step_1b": step_1b, "step_1c": step_1c,
    "step_2a": step_2a, "step_2b": step_2b,
    "step_3a": step_3a, "step_3b": step_3b, "step_3c": step_3c,
    "step_4a": step_4a, "step_4b": step_4b, "step_4c": step_4c,
    "step_5a": step_5a, "step_5b": step_5b, "step_5c": step_5c,
    "step_6a": step_6a, "step_6b": step_6b, "step_6c": step_6c,
    "joined": joined,
}

for name, df in artifacts.items():
    df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    print(f"  {name} → {df.shape}")

print("\nTerminé — fichiers dans", OUTPUT_DIR)

  step_1a → (13, 15)
  step_1b → (13, 15)
  step_1c → (13, 6)
  step_2a → (122, 16)
  step_2b → (155, 16)
  step_3a → (699, 9)
  step_3b → (862, 9)
  step_3c → (155, 63)
  step_4a → (7332, 10)
  step_4b → (8036, 10)
  step_4c → (155, 99)
  step_5a → (1253, 10)
  step_5b → (1319, 10)
  step_5c → (155, 11)
  step_6a → (859, 10)
  step_6b → (925, 10)
  step_6c → (155, 11)
  joined → (155, 188)

Terminé — fichiers dans /media/laghmari/ssd-data/dev/hotels/prepare/SalesPrep/Output


In [17]:
joined.to_excel("../Output/hotel_sales_data.xlsx", index = False)